#Code for training a CNN (resnet) to classify fruits and vegetables

In [2]:
#Import required packages
import kagglehub
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

/home/alkelly2/ece556/myvenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Hyperparmeters
batch_size = 10
learning_rate = 1e-4
num_epochs = 10
patience = 3 #Number of epochs to wait for improvement before stopping training

In [5]:
# Download latest version
path = kagglehub.dataset_download("sunnyagarwal427444/food-ingredient-dataset-51")

print("Path to dataset files:", path)

for root, dirs, files in os.walk(path):
    print(root)
    break

Resuming download from 1118830592 bytes (843743274 bytes left)...
Resuming download to /home/alkelly2/.cache/kagglehub/datasets/sunnyagarwal427444/food-ingredient-dataset-51/1.archive (1118830592/1962573866) bytes left.


100%|██████████| 1.83G/1.83G [00:17<00:00, 48.9MB/s]

Extracting files...


Path to dataset files: /home/alkelly2/.cache/kagglehub/datasets/sunnyagarwal427444/food-ingredient-dataset-51/versions/1
/home/alkelly2/.cache/kagglehub/datasets/sunnyagarwal427444/food-ingredient-dataset-51/versions/1


In [7]:
#Load the Dataset

train_dataset = datasets.ImageFolder(root=path+"/huggingface/Train")
val_dataset = datasets.ImageFolder(root=path+"/huggingface/val")

print("Number of classes:", len(train_dataset.classes))
print("Class names:", train_dataset.classes)

Number of classes: 51
Class names: ['Amaranth', 'Apple', 'Banana', 'Beetroot', 'Bell pepper', 'Bitter Gourd', 'Blueberry', 'Bottle Gourd', 'Broccoli', 'Cabbage', 'Cantaloupe', 'Capsicum', 'Carrot', 'Cauliflower', 'Chilli pepper', 'Coconut', 'Corn', 'Cucumber', 'Dragon_fruit', 'Eggplant', 'Fig', 'Garlic', 'Ginger', 'Grapes', 'Jalepeno', 'Kiwi', 'Lemon', 'Mango', 'Okra', 'Onion', 'Orange', 'Paprika', 'Pear', 'Peas', 'Pineapple', 'Pomegranate', 'Potato', 'Pumpkin', 'Raddish', 'Raspberry', 'Ridge Gourd', 'Soy beans', 'Spinach', 'Spiny Gourd', 'Sponge Gourd', 'Strawberry', 'Sweetcorn', 'Sweetpotato', 'Tomato', 'Turnip', 'Watermelon']


In [8]:
#Apply transform to the dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # ResNet standard
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    )
])

train_dataset.transform = transform #set the transform attribute of the ImageFolder class
val_dataset.transform = transform

#Create dataloaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

In [10]:
#Load Resnet model
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(train_dataset.classes)) #make the last fc layer output the correct number of classes

#Setup GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/home/alkelly2/ece556/myvenv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/alkelly2/ece556/myvenv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/alkelly2/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 52.1MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [11]:
#Train the model
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

#setup early exiting
best_loss = float('inf')
epochs_without_improvement = 0

#training loop
for epoch in range(num_epochs):
    print("Staring epoch", epoch)
    model.train() #enable training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images) #compute predictions
        loss = criterion(outputs, labels) #compute loss
        loss.backward() #backward pass
        optimizer.step() #update model weights

        #track training loss
        running_loss += loss.item()

        #track training accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    #calculate epoch loss
    epoch_loss = running_loss / len(train_loader)

    #calculate epoch accuracy
    epoch_acc = 100 * correct / total

    print("Epoch: ", epoch, "/", num_epochs, " Loss: ", epoch_loss, " Accuracy:", epoch_acc)
    
    #check if loss is still improving
    if epoch_loss < best_loss: #loss improved
        best_loss = epoch_loss
        epochs_without_improvement = 0
        
    else: #loss did not improve
        epochs_without_improvement += 1
        
    #stop training if loss is not improving
    if epochs_without_improvement >= patience:
        print("Early stopping triggered")
        break

Staring epoch 0


/home/alkelly2/ece556/myvenv/lib/python3.13/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch:  0 / 10  Loss:  1.6176188694952722  Accuracy: 63.709273182957396
Staring epoch 1
Epoch:  1 / 10  Loss:  0.5431635461392856  Accuracy: 87.5187969924812
Staring epoch 2
Epoch:  2 / 10  Loss:  0.2838306628485073  Accuracy: 93.48370927318295
Staring epoch 3
Epoch:  3 / 10  Loss:  0.16447110564207523  Accuracy: 96.74185463659148
Staring epoch 4
Epoch:  4 / 10  Loss:  0.12266106904953494  Accuracy: 97.4937343358396
Staring epoch 5
Epoch:  5 / 10  Loss:  0.10045671113708377  Accuracy: 97.69423558897243
Staring epoch 6
Epoch:  6 / 10  Loss:  0.105768897921958  Accuracy: 97.54385964912281
Staring epoch 7
Epoch:  7 / 10  Loss:  0.08246184236960565  Accuracy: 98.22055137844612
Staring epoch 8
Epoch:  8 / 10  Loss:  0.06290620342212119  Accuracy: 98.52130325814537
Staring epoch 9
Epoch:  9 / 10  Loss:  0.06701141362951457  Accuracy: 98.0701754385965


In [14]:
#Save trained model
PATH = './produce_net.pth'
torch.save(model.state_dict(), PATH)

In [13]:
#Find network validation accuracy
correct = 0
total = 0

model.eval() #enable evaluation mode

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images) #get predicted classes
        
        #compare predicted and actual labels
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        #get network accuracy
        val_acc = 100 * (correct / total)
        
    print("Validation accuracy of the network:", val_acc, "%")

Validation accuracy of the network: 97.65166340508806 %
